# Scraping WDumps data
<a href="https://colab.research.google.com/github/wmde/WDumps-scraper/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In order to better understand what kinds of entity data dump subsets our users are interested in, this repository scrapes all dump subsets listed under "recent dumps". The scrape includes a JSON representation of the filters that were used to generate the dump.

The notebook generates a csv file that includes filter data in a human-readable form. Each row of the csv includes the following columns:
- dump name
- URL
- filter (in human-readable form including labels for any items and properties used)
- statements included in the dump (in human-readable form)
- labels (yes/no)
- descriptions (yes/no)
- aliases (yes/no)
- sitelinks (yes/no)
- languages


## Preparing the workspace
### Installing dependencies

In [1]:
%pip install requests-cache

### Importing libraries

In [2]:
from bs4 import BeautifulSoup
import csv
import requests
import requests_cache

### Function definitions

In [3]:
def get(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return False

def get_cache(url):
  session = requests_cache.CachedSession('dump_cache')
  response = session.get(url)
  if response.status_code == 200:
    return response.text
  else:
     return False

def get_temp_cache(url):
  session = requests_cache.CachedSession('temp_cache', expire_after=7200)
  response = session.get(url)
  if response.status_code == 200:
    return response.text
  else:
     return False

def get_name(html_content):
  soup = BeautifulSoup(html_content, "html.parser")
  dump_header = soup.find("h2")
  if dump_header is not None:
    name = dump_header.text.split(": ", maxsplit=1)[-1]
    return name
  else:
    return ""

## Extract latest dump ID

In [4]:
start = "https://wdumps.toolforge.org/dumps"
html = get(start)
soup = BeautifulSoup(html, "html.parser")

link_elem = soup.find("table").find("a")
link_url = link_elem["href"]
last_id = int(link_url.split("/")[-1])
print(last_id)

5410


## Extract IDs and names
Save these in a dictionary with the URLs.

In [5]:
base_url = "https://wdumps.toolforge.org/dump/"
dump = {}
dumps = []

for i in range(1, last_id + 1):
  url = f"{base_url}{i}"
  if i > 5400:
    html = get_temp_cache(url)
  else:
    html = get_cache(url)
  if html:
    dump["url"] = url
    dump["id"] = i
    dump["name"] = get_name(html)
    dumps.append(dump.copy())

## Save data to csv

In [9]:
with open("POC-Wikidata-dumps.csv", "w", newline="") as csvfile:
  fieldnames = ["url", "id", "name"]
  writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
  writer.writeheader()

  for i in range(0, len(dumps)):
    writer.writerow({"url": dumps[i]["url"], "id": dumps[i]["id"], "name": dumps[i]["name"]})